# Assignment 3 — LLM & Embeddings

**Assessment:** 13 points, all autograded.

Every task here is tested **offline**: no network access is needed and none is allowed. That is
deliberate — a model response is not reproducible enough to grade against, so what is graded
is the code around it. The hidden tests use inputs beyond the ones shown here.

Each answer cell contains `raise NotImplementedError()` — replace it with your code and keep the function names and signatures unchanged.

## Setup — run first, do not modify

In [1]:
import json
import numpy as np

A = np.array([[1.0, 0.0, 0.0],
              [0.0, 2.0, 0.0],
              [1.0, 1.0, 0.0]], dtype=np.float32)
B = np.array([[1.0, 0.0, 0.0],
              [0.0, 0.0, 3.0]], dtype=np.float32)

SCHEMA = {
    "sentiment": ["positive", "neutral", "negative"],
    "topic": ["delivery", "payment", "product", "other"],
}

RESPONSES = [
    '{"sentiment": "negative", "topic": "delivery"}',
    '```json\n{"sentiment": "neutral", "topic": "other"}\n```',
    'Sure, here it is:\n{"sentiment": "positive", "topic": "product"}\nLet me know!',
    '```\n{"sentiment": "negative", "topic": "payment", "confidence": 0.8}\n```',
]
print("setup complete")

setup complete


## Task 1 — normalisation and cosine similarity (3 points)

Write `l2_normalise(M)` returning a copy of `M` whose rows have unit Euclidean length; a row of
all zeros must stay all zeros rather than produce `nan`. The input must not be modified.

Then write `cosine_matrix(A, B)` returning the matrix `S` with `S[i, j]` equal to the cosine
similarity between row `i` of `A` and row `j` of `B`. Use matrix multiplication, not a loop.

In [4]:
def l2_normalise( M ):
    copy = np.array( M, dtype=float, copy=True )
    lengths = np.linalg.norm( copy, axis=1, keepdims=True )
    return np.divide( copy ,lengths, out=copy, where=lengths != 0)

def cosine_matrix( A, B ):
    normalised_a = l2_normalise( A )
    normalised_b = l2_normalise( B )

    return normalised_a @ normalised_b.T

In [5]:
S = cosine_matrix(A, B)
assert S.shape == (3, 2)

## Task 2 — top-k retrieval (2 points)

Write `top_k(query, matrix, k)` where `query` is a one-dimensional vector and `matrix` has one
document per row. Return the **indices** of the `k` most similar rows by cosine similarity,
ordered from the most to the least similar. If `k` exceeds the number of rows, return all rows.
Ties may be broken arbitrarily.

In [6]:
def top_k( query, matrix, k ):
    query = np.asarray( query ).reshape( 1, -1 )
    matrix = np.asarray( matrix )
    similarities = cosine_matrix( query, matrix )[0]
    limit = min( k, matrix.shape[0] )
    sorted = np.argsort( similarities )
    descending = np.flip( sorted )
    return descending[:limit]

In [7]:
idx = top_k(np.array([1.0, 0.0, 0.0], dtype=np.float32), A, 2)
assert len(idx) == 2

## Task 3 — document vectors (2 points)

Write `mean_pool(vectors, weights=None)` returning one vector: the unweighted mean of the rows
when `weights` is `None`, otherwise the weighted mean with the given per-row weights. When the
weights sum to zero, return the unweighted mean.

In [10]:
def mean_pool( vectors, weights=None ):
    vectors = np.asarray( vectors )
    if weights is None:
        return np.mean( vectors, axis=0 )
    weights = np.asarray( weights )
    if np.sum( weights ) == 0:
        return np.mean( vectors, axis=0 )
    return np.average( vectors, axis=0, weights=weights )

In [11]:
v = mean_pool(A)
assert np.asarray(v).shape == (3,)

## Task 4 — surviving a model's output (4 points)

Write `extract_json(text)` returning the **first valid JSON object** in `text`. It must cope
with a bare object, a fenced block (with or without a language tag), surrounding prose, junk
braces before the object, and a second object after it (which is ignored). Raise `ValueError`
when there is no JSON object at all.

Write `validate(record, schema)` returning a dictionary containing **exactly** the schema's keys
in the schema's order. Raise `ValueError` if a key is missing or a value is outside the allowed
list; silently drop keys the schema does not mention.

In [12]:
def extract_json( text ):
    decoder = json.JSONDecoder()
    for start, character in enumerate( text ):
        if character != "{":
            continue
        try:
            record, _ = decoder.raw_decode( text, start )
        except json.JSONDecodeError:
            continue
        if isinstance( record, dict ):
            return record
    raise ValueError( "No JSON object found" )

def validate( record, schema ):
    missing = set( schema ) - set( record )
    if missing:
        raise ValueError( f"Missing keys: {sorted( missing )}" )
    for key, allowed_values in schema.items():
        if record[key] not in allowed_values:
            raise ValueError( f"{key}={record[key]!r} not in {allowed_values}" )
    return { key: record[key] for key in schema }

In [13]:
r = extract_json(RESPONSES[0])
assert isinstance(r, dict)

## Task 5 — Plan a rate-limit retry (2 points)

Implement `retry_delay(status, attempt, retry_after=None, quota_exhausted=False)`.
This is a deterministic planning function: do not sleep or make network requests.

- Return `None` if `status` is not HTTP `429`, or if `quota_exhausted` is `True`.
- Otherwise, exponential backoff is `2 ** attempt` seconds, capped at `32` seconds.
- If `retry_after` is supplied, return the larger of that value and the capped backoff.
  The server's requested delay may exceed 32 seconds.

`status` is an integer, `attempt` is a non-negative integer (zero for the first retry),
`retry_after` is `None` or a finite non-negative number of seconds, and `quota_exhausted`
is a boolean. HTTP-date parsing, jitter, retry limits and rate limiting are outside this
function's scope; a real client needs those policies too. Waiting does not restore an exhausted quota.

In [14]:
def retry_delay( status, attempt, retry_after=None, quota_exhausted=False ):
    if status != 429 or quota_exhausted:
        return None
    backoff = min( 2 ** attempt, 32 )
    if retry_after is None:
        return backoff
    return max( retry_after, backoff )

In [15]:
assert retry_delay(429, 0) == 1
assert retry_delay(429, 2, retry_after=10) == 10
assert retry_delay(200, 0) is None